**Created: 2026-11-28**

# Purcell filter from GDS: HFSS eigenmode

An HFSS eigenmode simulation of the Purcell-filter readout sample in `readout_purcell_800mhz.gds`: the two-port quarter-wave filter (target 7.5 GHz center, 800 MHz bandwidth), its five readout resonators and their claws, and the W4 / W1 launchpads. The layout is the one `readout.ipynb` builds.

**What this notebook does:**

1. Reads the GDS into a single Qiskit Metal component (`GdsLayout`, defined below). The GDS draws metal (layer 5, M1), so the component turns it back into what Metal renders: the holes in the ground plane as etched pockets, and any metal not connected to ground (here the W4 pad, its lead and the input side of the interdigital capacitor) as metal islands.
2. Finds the launchpads (pockets that reach to within `pad_edge_distance_um` of the chip edge) and puts a pin on the back of each pad. Each pin becomes a 50 ohm port in HFSS: a lumped RLC sheet from the pad's back edge across the etched gap to ground. In eigenmode, this load sets each mode's Q.
3. Runs HFSS eigenmode: 6 modes from 6 GHz, auto mesh.

**Notes:**
* HFSS eigenmode only takes a lowest frequency, not a highest one. It finds the 6 modes just above 6 GHz. The expected modes are the filter plus five readouts at 7.3–7.7 GHz. Any mode above 8 GHz is flagged in the results table.
* Auto mesh means HFSS's own initial mesh (mesh method *Auto*) plus adaptive refinement. The extra mesh-length operation that Metal puts on eigenmode ports is removed (set `hfss_keep_port_mesh = True` to keep it).
* The substrate is MIT LL BISQuit8 (725 um silicon, eps_r 11.45), as in `readout.ipynb`.

**To change the setup:** edit the *Parameters* cell, then **Restart & Run All**.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gdstk
from shapely.geometry import LineString, MultiPolygon, Polygon, box
from shapely.ops import unary_union

os.environ.setdefault("QISKIT_METAL_HEADLESS", "True")

from qiskit_metal import designs, Dict
import qiskit_metal as qm
from qiskit_metal.qlibrary.core import QComponent

## Parameters

In [ ]:
# --- Layout ------------------------------------------------------------
gds_file = "readout_purcell_800mhz.gds"
gds_cell = ""  # "" = the file's top cell
gds_metal_layer = (5, 0)  # (layer, datatype) of the metal (MIT LL M1)
gds_simplify_um = 0.1  # drop outline points closer than this to a straight line
pad_edge_distance_um = 250  # a pocket this close to the chip edge holds a launchpad

# --- Chip (MIT LL BISQuit8, as in readout.ipynb) -------------------------
chip_size_mm = 5.0
substrate_thickness_um = 725
substrate_eps_r = 11.45
substrate_material = "silicon_bisquit"

# --- Filter targets (for the results table) ---------------------------------
resonator_freqs_ghz = [7.3, 7.4, 7.5, 7.6, 7.7]  # from readout.ipynb
purcell_filter_freq_ghz = 7.5
purcell_filter_bw_ghz = 0.8

# --- HFSS eigenmode -----------------------------------------------------------
hfss_n_modes = 6
hfss_min_freq_ghz = 6.0
hfss_max_freq_ghz = 8.0  # not an HFSS setting: modes above this are flagged
port_impedance_ohm = 50
hfss_max_passes = 20
hfss_min_passes = 2
hfss_min_converged = 2  # passes in a row that must meet max_delta_f
hfss_max_delta_f = 0.1  # % change in frequency per pass
hfss_pct_refinement = 30  # % of mesh elements refined per pass
hfss_basis_order = -1  # 0 zero, 1 first, 2 second, -1 mixed
hfss_keep_port_mesh = False  # True keeps Metal's 7 um mesh-length seed on the ports
# Ground plane: the bounding box of the layout plus these buffers. 0.1 mm in
# x puts the west edge of the ground at the chip edge (the pad pockets reach
# to 100 um from it). Set hfss_full_chip = True to draw the whole 5 x 5 mm chip.
hfss_x_buffer_mm = 0.1
hfss_y_buffer_mm = 0.3
hfss_full_chip = False

run_hfss = True
hfss_project_file = "readout_gds_eigenmode.aedt"  # new each run; _2, _3... if taken
hfss_results_dir = "hfss_results/readout_gds"

## GDS to Metal

`GdsLayout` reads one layer of a GDS cell as a Metal component.

* **Ground plane:** the largest piece of metal (the one reaching the chip edge). Its holes become etched pockets (`subtract=True`).
* **Metal islands:** every other piece of metal is drawn on top of the pocket that holds it.
* **Pads:** each straight stretch of pocket edge that faces the chip edge, within `pad_edge_distance_um` of it, is the etch behind a launchpad. A ray from its middle toward the chip centre finds the pad's back face. The pin there is as wide as the pad, its normal points at the chip edge, and its gap is the etch between pad and ground. The pins are named `pad_<edge>_<letter>`, lettered south to north (west to east on the N/S edges): here `pad_W_a` is the southern pad (LP_W1, output) and `pad_W_b` the northern one (LP_W4, input). Letters, not numbers, so they are not confused with the package's pad numbers.

In [ ]:
class GdsLayout(QComponent):
    """One metal layer of a GDS cell, as etched pockets in Metal's ground plane
    plus floating metal islands, with a pin on the back of every launchpad.

    Options:
        * gds_file, gds_cell ("" = top cell), gds_layer, gds_datatype
        * simplify_um: outline simplification tolerance
        * pad_edge_distance_um: how close to the chip edge a pad pocket reaches
    """

    default_options = Dict(gds_file="", gds_cell="", gds_layer=5, gds_datatype=0,
                           simplify_um=0.1, pad_edge_distance_um=250)
    component_metadata = Dict(short_name="GDS", _qgeometry_table_poly="True")

    def make(self):
        o = self.options  # raw strings: the file name must not go through Metal's unit parser
        lib = gdstk.read_gds(str(o.gds_file))
        cell = lib[str(o.gds_cell)] if o.gds_cell else lib.top_level()[0]
        flat = cell.copy(cell.name + "_flat").flatten()
        to_mm = lib.unit / 1e-3
        layer = (int(o.gds_layer), int(o.gds_datatype))
        polygons = [Polygon(p.points * to_mm) for p in flat.polygons
                    if (p.layer, p.datatype) == layer]
        if not polygons:
            raise ValueError(f"No polygons on layer {layer} of {cell.name}.")

        # The GDS cuts the ground plane into pieces (polygons with holes are
        # split); the union joins them again.
        metal = unary_union(polygons)
        parts = list(metal.geoms) if isinstance(metal, MultiPolygon) else [metal]
        ground = max(parts, key=lambda g: g.area)
        islands = [g for g in parts if g is not ground]
        pockets = [Polygon(ring) for ring in ground.interiors]
        tol = float(o.simplify_um) / 1e3
        self.add_qgeometry("poly", {f"pocket{i}": g.simplify(tol) for i, g in enumerate(pockets)},
                           subtract=True)
        if islands:
            self.add_qgeometry("poly", {f"island{i}": g.simplify(tol) for i, g in enumerate(islands)})
        self.n_pockets, self.n_islands = len(pockets), len(islands)

        # Launchpads.
        x0, y0, x1, y1 = ground.exterior.bounds
        near = float(o.pad_edge_distance_um) / 1e3
        # edge: (axis across it, outward sign, chip edge coordinate)
        sides = {"W": (0, -1, x0), "E": (0, 1, x1), "S": (1, -1, y0), "N": (1, 1, y1)}
        found = []
        for pocket in pockets:
            ring = np.asarray(pocket.simplify(0).exterior.coords)
            for edge, (a, s, chip_end) in sides.items():
                pocket_end = pocket.bounds[a] if s < 0 else pocket.bounds[a + 2]
                if abs(chip_end - pocket_end) > near:
                    continue
                for p, q in zip(ring[:-1], ring[1:]):
                    if abs(p[a] - pocket_end) > 1e-6 or abs(q[a] - pocket_end) > 1e-6:
                        continue  # not a stretch of the pocket's outer edge
                    normal = np.zeros(2)
                    normal[a] = s  # toward the chip edge
                    mid = (p + q) / 2
                    hit = LineString([mid, mid - normal * 2 * near]).intersection(metal.boundary)
                    if hit.is_empty:
                        continue
                    hits = np.array([pt.coords[0] for pt in getattr(hit, "geoms", [hit])])
                    dist = np.abs(hits[:, a] - pocket_end)
                    dist = dist[dist > 1e-6]  # not the pocket's own edge
                    if not dist.size:
                        continue
                    gap = dist.min()
                    face = pocket_end - s * gap
                    # The pad's width: the metal just inside its back face.
                    lo, hi = sorted((p[1 - a], q[1 - a]))
                    depth = sorted((face - s * 1e-5, face - s * 1e-3))
                    strip = ([depth[0], lo + 1e-5, depth[1], hi - 1e-5] if a == 0
                             else [lo + 1e-5, depth[0], hi - 1e-5, depth[1]])
                    b = metal.intersection(box(*strip)).bounds
                    lo, hi = (b[1], b[3]) if a == 0 else (b[0], b[2])
                    centre = np.zeros(2)
                    centre[a], centre[1 - a] = face, (lo + hi) / 2
                    found.append((edge, centre, normal, hi - lo, gap))

        self.pads = []
        for edge, (a, _, _) in sides.items():
            on_edge = sorted((f for f in found if f[0] == edge), key=lambda f: f[1][1 - a])
            for n, (_, centre, normal, width, gap) in enumerate(on_edge):
                # Metal's pin normal is the pin line turned 90 degrees anticlockwise.
                tangent = np.array([normal[1], -normal[0]])
                name = f"pad_{edge}_{'abcdefghij'[n]}"
                self.add_pin(name, np.array([centre - tangent * width / 2,
                                             centre + tangent * width / 2]), width, gap=gap)
                self.pads.append(name)

## Design

In [ ]:
design = designs.DesignPlanar()
design.chips.main.size.size_x = f"{chip_size_mm}mm"
design.chips.main.size.size_y = f"{chip_size_mm}mm"
design.chips.main.size.size_z = f"-{substrate_thickness_um}um"
design.chips.main.material = substrate_material  # made in AEDT by the HFSS cell
design.overwrite_enabled = True

layout = GdsLayout(design, "Readout_GDS", options=dict(
    gds_file=gds_file, gds_cell=gds_cell,
    gds_layer=gds_metal_layer[0], gds_datatype=gds_metal_layer[1],
    simplify_um=gds_simplify_um, pad_edge_distance_um=pad_edge_distance_um,
))

print(f"{gds_file}: {layout.n_pockets} etched pockets, {layout.n_islands} metal islands")
print("Launchpads (each a port):")
for name in layout.pads:
    pin = layout.pins[name]
    print(f"  {name}: back face at ({pin.middle[0]:.3f}, {pin.middle[1]:.3f}) mm, "
          f"{pin.width * 1e3:.0f} um wide, {pin.gap * 1e3:.0f} um etch to ground")
if len(layout.pads) != 2:
    print(f"WARNING: expected 2 launchpads (W4 and W1), found {len(layout.pads)}.")

In [ ]:
fig = qm.view(design)
fig.suptitle(gds_file)
ax = fig.axes[0]
for name in layout.pads:
    x, y = layout.pins[name].middle
    ax.annotate(name, (x, y), (x + 0.6, y), color="tab:red", va="center",
                arrowprops=dict(arrowstyle="->", color="tab:red"))
fig

## HFSS eigenmode

Runs in Ansys HFSS (AEDT 2024 R2) through Metal's `hfss` renderer, which starts AEDT if no 2024 R2 copy is open. Each run starts a new AEDT project, saved next to this notebook as `hfss_project_file` (or with `_2`, `_3`, ... added if that file exists).

Each launchpad is terminated in `port_impedance_ohm`, so every mode's Q is its leakage out of the two ports:
* the filter: the lowest-Q mode, Q ~ f / bandwidth (7.5 / 0.8 ~ 9.4 at target)
* each readout: kappa / 2 pi = f / Q

In [ ]:
if run_hfss:
    import subprocess
    from qiskit_metal.analyses.simulation import EigenmodeSim

    stem, ext = os.path.splitext(os.path.abspath(hfss_project_file))
    hfss_project_path, k = stem + ext, 2
    while os.path.exists(hfss_project_path):
        hfss_project_path, k = f"{stem}_{k}{ext}", k + 1
    hfss = design.renderers.hfss
    # Attach to an AEDT 2024 R2 that is already open; start one only if not.
    running = subprocess.run(
        ["powershell", "-NoProfile", "-Command",
         "(Get-Process ansysedt -ErrorAction SilentlyContinue).Path"],
        capture_output=True, text=True,
    ).stdout
    if "v242" not in running:
        hfss.open_ansys(path_var="ANSYSEM_ROOT242")
    hfss.start()
    hfss.rdesktop.new_project().save(hfss_project_path)
    hfss.options.project_name = os.path.splitext(os.path.basename(hfss_project_path))[0]
    hfss.options.project_path = None
    hfss.start()  # reconnect, now to the new project
    print(f"HFSS project: {hfss_project_path} (AEDT {hfss.rdesktop.get_version()})")
    hfss.pinfo.project._project.GetDefinitionManager().AddMaterial(
        [f"NAME:{substrate_material}", "CoordinateSystemType:=", "Cartesian",
         "BulkOrSurfaceType:=", 1, ["NAME:PhysicsTypes", "set:=", ["Electromagnetic"]],
         "permittivity:=", str(substrate_eps_r), "dielectric_loss_tangent:=", "0"]
    )
    print(f"Substrate: {substrate_material}, eps_r {substrate_eps_r}, {substrate_thickness_um} um")

In [ ]:
def use_auto_mesh(renderer):
    """Auto mesh: HFSS's own initial mesh (method Auto), no seeding."""
    mesh = renderer.pinfo.design._mesh  # the design's MeshSetup module
    if not hfss_keep_port_mesh:
        try:
            mesh.DeleteOp(["port_mesh"])  # Metal's mesh-length operation on the ports
        except Exception:
            pass  # none was made
    mesh.InitialMeshSettings([
        "NAME:MeshSettings",
        ["NAME:GlobalSurfApproximation", "CurvedSurfaceApproxChoice:=", "UseSlider",
         "SliderMeshSettings:=", 5],
        ["NAME:GlobalCurvilinear", "Apply:=", False],
        ["NAME:GlobalModelRes", "UseAutoLength:=", True],
        "MeshMethod:=", "Auto",
        "UseLegacyFaceterForTauVolumeMesh:=", False,
        "DynamicSurfaceResolution:=", False,
        "UseFlexMeshingForTAUvolumeMesh:=", False,
        "UseAlternativeMeshMethodsAsFallBack:=", True,
        "AllowPhiForLayeredGeometry:=", False,
    ])
    print("Mesh: Auto initial mesh, adaptive refinement"
          + (" (plus Metal's port mesh)" if hfss_keep_port_mesh else ""))


if run_hfss:
    eig = EigenmodeSim(design, "hfss")
    eig.renderer.options.x_buffer_width_mm = hfss_x_buffer_mm
    eig.renderer.options.y_buffer_width_mm = hfss_y_buffer_mm
    eig.setup.update(
        n_modes=hfss_n_modes,
        min_freq_ghz=hfss_min_freq_ghz,
        max_passes=hfss_max_passes,
        min_passes=hfss_min_passes,
        min_converged=hfss_min_converged,
        max_delta_f=hfss_max_delta_f,
        pct_refinement=hfss_pct_refinement,
        basis_order=hfss_basis_order,
    )

    # EigenmodeSim renders the design, then solves (_analyze). Set the mesh
    # in between, once the design exists in AEDT.
    solve = eig._analyze
    def mesh_then_solve():
        use_auto_mesh(eig.renderer)
        solve()
    eig._analyze = mesh_then_solve

    eig.run(
        name="Purcell_GDS_eigenmode",
        components=["Readout_GDS"],
        open_terminations=[],
        port_list=[("Readout_GDS", pad, port_impedance_ohm) for pad in layout.pads],
        box_plus_buffer=not hfss_full_chip,
    )
    eig.plot_convergences()

In [ ]:
if run_hfss:
    freqs, kappas = eig.renderer.pinfo.setup.get_solutions().eigenmodes()
    f_ghz = np.asarray(freqs, dtype=float)
    k_ghz = np.asarray(kappas if kappas is not None else [np.nan] * len(f_ghz), dtype=float)
    if np.nanmax(f_ghz) > 1e3:  # exported in Hz
        f_ghz, k_ghz = f_ghz / 1e9, k_ghz / 1e9
    q = f_ghz / k_ghz

    # Names: the lowest-Q mode in the 6-8 GHz window is the filter; the rest
    # in the window are the readouts in frequency order, then anything extra.
    order = list(np.argsort(f_ghz))
    in_band = [m for m in order if hfss_min_freq_ghz <= f_ghz[m] <= hfss_max_freq_ghz]
    n_res = len(resonator_freqs_ghz)
    labels = {}
    if in_band:
        filt = min(in_band, key=lambda m: q[m])
        labels[filt] = "PurcellFilter"
        readouts = [m for m in in_band if m != filt]
        for i, m in enumerate(readouts):
            labels[m] = f"ReadoutRes{i}" if i < n_res else f"extra{i - n_res}"
    for m in order:
        labels.setdefault(m, f"above_{hfss_max_freq_ghz:g}GHz_mode{m + 1}")

    targets = {f"ReadoutRes{i}": f for i, f in enumerate(resonator_freqs_ghz)}
    targets["PurcellFilter"] = purcell_filter_freq_ghz
    modes = pd.DataFrame(
        {
            labels[m]: {
                "mode": m + 1,
                "f (GHz)": f_ghz[m],
                "target (GHz)": targets.get(labels[m], np.nan),
                "Q": q[m],
                "kappa/2pi (MHz)": k_ghz[m] * 1e3,
            }
            for m in order
        }
    ).T
    os.makedirs(hfss_results_dir, exist_ok=True)
    modes.to_csv(os.path.join(hfss_results_dir, "Purcell_GDS_eigenmode_modes.csv"))
    print(f"Eigenmodes, both pads terminated in {port_impedance_ohm} ohms:")
    print(modes.to_string(float_format=lambda v: f"{v:.4g}"))

    if "PurcellFilter" in modes.index:
        filt = modes.loc["PurcellFilter"]
        print(f"\nFilter: {filt['f (GHz)']:.3f} GHz, bandwidth f/Q = {filt['kappa/2pi (MHz)']:.0f} MHz "
              f"(target {purcell_filter_freq_ghz} GHz, {purcell_filter_bw_ghz * 1e3:.0f} MHz)")
    outside = [labels[m] for m in order if not hfss_min_freq_ghz <= f_ghz[m] <= hfss_max_freq_ghz]
    if outside:
        print(f"Outside {hfss_min_freq_ghz:g}-{hfss_max_freq_ghz:g} GHz: {', '.join(outside)}")

In [ ]:
# Save the AEDT project and release the connection. AEDT stays open for inspection.
if run_hfss:
    hfss.pinfo.project.save()
    print(f"Saved the HFSS project to {hfss_project_path}")
    hfss.stop()